In [32]:
#Phase 2 - Task 2: Model Optimization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
 
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
 
RAW_PATH = "../data/heart.csv"
PLOTS_DIR = "../output/heart_plots"
MODELS_DIR = "../output/heart_models"
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
 

In [33]:
# 1. LOAD & CLEAN RAW DATA
df = pd.read_csv(RAW_PATH)
df = df.drop_duplicates().reset_index(drop=True)
 
continuous_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
nominal_cols = ["cp", "restecg", "slope", "thal"]
passthrough_cols = ["sex", "fbs", "exang", "ca"]
 
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr
 
for col in continuous_cols:
    low, high = iqr_bounds(df[col])
    df[col] = df[col].clip(lower=low, upper=high)
 
X = df.drop(columns=["target"])
y = df["target"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [34]:
# 2. PREPROCESSING PIPELINE
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), continuous_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), nominal_cols),
        ("bin", "passthrough", passthrough_cols),
    ]
)
 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [35]:
# 3. HYPERPARAMETER GRIDS FOR TOP CANDIDATE MODELS
param_grids = {
    "Logistic Regression": {
        "estimator": LogisticRegression(max_iter=2000, random_state=42),
        "params": {
            "clf__C": [0.01, 0.1, 0.5, 1, 5, 10],
            "clf__penalty": ["l2"],
            "clf__solver": ["lbfgs", "liblinear"],
        },
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(random_state=42),
        "params": {
            "clf__n_estimators": [100, 200, 300],
            "clf__max_depth": [3, 5, 7, None],
            "clf__min_samples_split": [2, 5, 10],
            "clf__min_samples_leaf": [1, 2, 4],
        },
    },
    "SVM (RBF)": {
        "estimator": SVC(probability=True, random_state=42),
        "params": {
            "clf__C": [0.1, 1, 5, 10],
            "clf__gamma": ["scale", "auto", 0.01, 0.1],
            "clf__kernel": ["rbf"],
        },
    },
    "KNN": {
        "estimator": KNeighborsClassifier(),
        "params": {
            "clf__n_neighbors": [3, 5, 7, 9, 11, 13],
            "clf__weights": ["uniform", "distance"],
            "clf__p": [1, 2],
        },
    },
    "Decision Tree": {
        "estimator": DecisionTreeClassifier(random_state=42),
        "params": {
            "clf__max_depth": [3, 4, 5, 6, 8, None],
            "clf__min_samples_split": [2, 5, 10],
            "clf__criterion": ["gini", "entropy"],
        },
    },
    "Naive Bayes": {
        "estimator": GaussianNB(),
        "params": {"clf__var_smoothing": np.logspace(-11, -7, 5)},
    },
}


In [36]:
# 4. GRID SEARCH + CROSS-VALIDATION FOR EACH MODEL
tuned_results = []
best_estimators = {}
 
for name, cfg in param_grids.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", cfg["estimator"])])
    grid = GridSearchCV(pipe, cfg["params"], cv=cv, scoring="roc_auc", n_jobs=-1)
    grid.fit(X_train, y_train)
 
    best_pipe = grid.best_estimator_
    best_estimators[name] = best_pipe
 
    # 5-fold CV score of the tuned model on training data
    cv_scores = cross_val_score(best_pipe, X_train, y_train, cv=cv, scoring="accuracy")
 
    # Test set evaluation
    y_pred = best_pipe.predict(X_test)
    y_proba = best_pipe.predict_proba(X_test)[:, 1]
 
    tuned_results.append({
        "Model": name,
        "Best Params": str(grid.best_params_),
        "CV Accuracy (mean)": cv_scores.mean(),
        "CV Accuracy (std)": cv_scores.std(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred),
        "Test Recall": recall_score(y_test, y_pred),
        "Test F1-Score": f1_score(y_test, y_pred),
        "Test ROC-AUC": roc_auc_score(y_test, y_proba),
    })
    print(f"\n{name}: best CV ROC-AUC={grid.best_score_:.4f} | best params={grid.best_params_}")
 
tuned_df = pd.DataFrame(tuned_results).sort_values("Test ROC-AUC", ascending=False).reset_index(drop=True)
print("\n=== Tuned Model Comparison ===")
print(tuned_df.drop(columns=["Best Params"]).round(4))
tuned_df.to_csv("../output/tuned_model_comparison.csv", index=False)
 


Logistic Regression: best CV ROC-AUC=0.9093 | best params={'clf__C': 1, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}

Random Forest: best CV ROC-AUC=0.9066 | best params={'clf__max_depth': 7, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 5, 'clf__n_estimators': 100}

SVM (RBF): best CV ROC-AUC=0.9028 | best params={'clf__C': 10, 'clf__gamma': 0.01, 'clf__kernel': 'rbf'}

KNN: best CV ROC-AUC=0.8959 | best params={'clf__n_neighbors': 11, 'clf__p': 1, 'clf__weights': 'distance'}

Decision Tree: best CV ROC-AUC=0.8280 | best params={'clf__criterion': 'entropy', 'clf__max_depth': 3, 'clf__min_samples_split': 2}

Naive Bayes: best CV ROC-AUC=0.8595 | best params={'clf__var_smoothing': 1e-11}

=== Tuned Model Comparison ===
                 Model  CV Accuracy (mean)  CV Accuracy (std)  Test Accuracy  \
0            SVM (RBF)              0.8463             0.0571         0.8361   
1          Naive Bayes              0.8088             0.1093         0.8197   
2                 

In [37]:
# 5. SELECT & SAVE BEST MODEL 
# NOTE: SVM (RBF) has the single highest ROC-AUC (0.909 vs 0.894), but the
# margin is marginal and Logistic Regression outperforms it on Accuracy,
# Precision, and F1-Score while remaining fully interpretable (coefficients
# map directly to clinical risk factors). Since this project requires
# generating clinical insights and health recommendations from the model,
# interpretability is a deployment requirement, not a nice-to-have -> we
# select Logistic Regression as the final deployed model.
best_model_name = "Logistic Regression"
best_pipeline = best_estimators[best_model_name]
joblib.dump(best_pipeline, f"{MODELS_DIR}/heart_disease_best_pipeline.pkl")
 
best_row = tuned_df[tuned_df["Model"] == best_model_name].iloc[0]
with open(f"{MODELS_DIR}/best_model_info.json", "w") as f:
    json.dump({
        "best_model": best_model_name,
        "test_accuracy": float(best_row["Test Accuracy"]),
        "test_roc_auc": float(best_row["Test ROC-AUC"]),
        "feature_columns": list(X.columns),
        "continuous_cols": continuous_cols,
        "nominal_cols": nominal_cols,
        "passthrough_cols": passthrough_cols,
    }, f, indent=2)
 
print(f"\nBest model selected: {best_model_name}")
print(f"Saved deployable pipeline to {MODELS_DIR}/heart_disease_best_pipeline.pkl")


Best model selected: Logistic Regression
Saved deployable pipeline to ../output/heart_models/heart_disease_best_pipeline.pkl


In [38]:
# 6. VISUALIZE: TUNED MODEL COMPARISON
plt.figure(figsize=(10, 6))
melted = tuned_df.melt(
    id_vars="Model",
    value_vars=["Test Accuracy", "Test Precision", "Test Recall", "Test F1-Score", "Test ROC-AUC"],
)
sns.barplot(data=melted, x="Model", y="value", hue="variable")
plt.xticks(rotation=20)
plt.ylim(0, 1.05)
plt.title("Tuned Model Performance Comparison (Post Hyperparameter Optimization)")
plt.ylabel("Score")
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/tuned_model_comparison_bar.png")
plt.close()

In [39]:
# 7. ROC CURVES FOR ALL TUNED MODELS
plt.figure(figsize=(7, 7))
for name, pipe in best_estimators.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Tuned Models")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/roc_curves_tuned.png")
plt.close()
 

In [40]:
# 8. CONFUSION MATRIX FOR BEST MODEL
y_pred_best = best_pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Disease", "Disease"], yticklabels=["No Disease", "Disease"])
plt.title(f"Confusion Matrix - Best Model ({best_model_name})")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/best_model_confusion_matrix.png")
plt.close()

In [41]:
# 9. FEATURE IMPORTANCE / COEFFICIENTS FOR BEST MODEL
feature_names = best_pipeline.named_steps["prep"].get_feature_names_out()
feature_names = [f.split("__")[-1] for f in feature_names]
clf = best_pipeline.named_steps["clf"]
 
plt.figure(figsize=(8, 7))
if hasattr(clf, "feature_importances_"):
    imp_csv = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=False)
    sns.barplot(x=imp_csv.values, y=imp_csv.index, color="#3498db")
    plt.xlabel("Importance")
elif hasattr(clf, "coef_"):
    imp_csv = pd.Series(clf.coef_[0], index=feature_names).sort_values(key=abs, ascending=False)
    sns.barplot(x=imp_csv.values, y=imp_csv.index, color="#3498db")
    plt.xlabel("Coefficient (log-odds impact)")
else:
    # Model has no built-in importance (e.g. SVM with RBF kernel) -> use permutation importance
    from sklearn.inspection import permutation_importance
    perm = permutation_importance(best_pipeline, X_test, y_test, n_repeats=30, random_state=42, scoring="roc_auc")
    imp_csv = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
    sns.barplot(x=imp_csv.values, y=imp_csv.index, color="#3498db")
    plt.xlabel("Permutation Importance (mean ROC-AUC drop)")
plt.title(f"Feature Importance - {best_model_name}")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/best_model_feature_importance.png")
plt.close()
imp_csv.to_csv("../output/best_model_feature_importance.csv", header=["importance"])
 
print("\nPhase 2 optimization complete. All outputs saved to ../outputs/")


Phase 2 optimization complete. All outputs saved to ../outputs/
